# SAIGE GWAS

## Imports


In [ ]:
import os
from datetime import date
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed
import subprocess
import numpy as np
import pandas as pd
import polars as pl
import gwaslab as gl
import matplotlib.pyplot as plt

In [ ]:
d = date.today()
t = datetime.now().time()

print(f'''
DATE: {d} at {t}

pandas=={pd.__version__}
numpy=={np.__version__}
''')

## Set directories and variables

#### Common paths

In [ ]:
# Directories

# Hestia NGS Software
tools = "/path/to/tools"

# Main directory
main_dir = "/path/to/home"
MAIN_DIR = main_dir     ### alias

# Data directory
data_dir = f"{main_dir}/data"
DATA_DIR = data_dir     ### alias

# Raw data directory
raw_dir = f"{data_dir}/RAW"
RAW_DIR = raw_dir     ### alias

# Imputed data directory
impt_dir = f"{data_dir}/IMPUTED"
IMPT_DIR = raw_dir     ### alias

# Imputed/softcalls data directory
soft_dir = f"{data_dir}/IMPUTED/softcalls"
SOFT_DIR = raw_dir     ### alias

# Typed onlye data directory
typed_dir = f"{data_dir}/IMPUTED/typed"
TYPED_DIR = raw_dir     ### alias

# Meta data (covariate, population, ancestry labels, etc.)
meta_dir = f"{data_dir}/META"
META_DIR = meta_dir     ### alias

In [ ]:
## Create SAIGE directories

# Projected PCA directory
pca_dir = f"{analysis}/PCA"
PCA_DIR = pca_dir     ### alias
# os.makedirs(pca_dir, exist_ok=True)

# GRM directory
grm_dir = f"{analysis}/GRM"
GRM_DIR = grm_dir     ### alias
os.makedirs(grm_dir, exist_ok=True)

# SAIGE Step 1
step1 = f"{analysis}/SAIGE_STEP1"
STEP1 = step1     ### alias
os.makedirs(step1, exist_ok=True)

# SAIGE Step 2
step2 = f"{analysis}/SAIGE_STEP2"
STEP2 = step2     ### alias
os.makedirs(step2, exist_ok=True)

# SAIGE GWAS
gwas_dir = f"{analysis}/SAIGE_GWAS"
GWAS_DIR = gwas_dir     ### alias
os.makedirs(gwas_dir, exist_ok=True)

#### Paths to software and tools

In [ ]:
# SAIGE scripts path
saige = f"{tools}/SAIGE_pixi/SAIGE/extdata"

# Plink1.9 and Plink2.0 path
plink = f"{tools}/plink_linux_x86_64_20250615/plink"
plink2 = f"{tools}/plink2_linux_avx2_20250609/plink2"

# KING path
king = f"{tools}/king/king"

# GCTA
gcta = f"{tools}/gcta_1.93.1beta/gcta64"

# PCA Projected path
pca_path = f"{tools}/ProjectedPCAAndModelSelection"

#### Input, output, covariate files

In [ ]:
# Covariate file
covar = f"{meta_dir}/CATPD.cov"

# Populations/Ancestries file
pops = f"{meta_dir}/CATPD.pop"

# Pheno Name
pheno = f"DISEASE"

# Chromosomes as list
chromosomes = list(range(1,23))

# Output directory path and output prefix
output = "CATPD"

In [ ]:
# Empty file for remove samples if there's no outlier list
remove = pd.DataFrame([["NA", "NA"]])
samplesToRemove = f"{meta_dir}/samplestoremove.txt"
remove.to_csv(samplesToRemove, sep="\t", index=False, header=False, na_rep='NA')

## Run SAIGE

#### Define input variables

In [ ]:
# Input file: imputed and phased typed variants
inputPfile = f"{typed_dir}/chr1_22.typed.vcf.gz",

# Covariate file and covariates
covar_grm = covar_grm_path

covariate_list = ["SEX","AGE"] + ["GCTA_PC" + str(i) for i in range(1, 11)]

qcovariate_list = ["SEX"]

# Phenotype 
pheno = "DISEASE"
# Sample ID
sampleID = "IID"

# Prefix 
prefix = "CATPD"
# Relatedness cutoff
rel_cutoff = "0.0884"
# Kinship marker
kin_marker = "500000"

# Related file
related_file = f"{prefix}_relatednessCutoff_{rel_cutoff}_{kin_marker}_randomMarkersUsed"

# Threads
threads = str(1)

Create a plink2 file from the vcf.gz file

In [ ]:
# Convert vcf to Plink1.9 and Plink2.0 files
vcf2pgen = [
    "plink2",
    "--vcf", f"{typed_dir}/chr1_22.typed.vcf.gz",
    "--psam", f"{inputPfile}.psam",
    "--keep-allele-order",
    "--make-pgen",
    "--sort-vars",
    "--silent",
    "--threads", str(threads),
    "--out", f"{typed_dir}/chr1_22.typed",
]

pgen2bed = [
    "plink2",
    "--pfile", f"{typed_dir}/chr1_22.typed",
    "--keep-allele-order",
    "--make-bed",
    "--sort-vars",
    "--silent",
    "--threads", str(threads),
    "--out", f"{typed_dir}/chr1_22.typed",
]

# Run processes
subprocess.run(vcf2pgen, check=True)
subprocess.run(pgen2bed, check=True)

### Get Projected PCA

In [ ]:
# Get Projected PCs
createProjectedPCA = ["python3", f"{pca_path}/covarProjectedPCA.py",
                    "-A", inputPfile,             # Should be plink1.9 bfile 
                    "-t", covar_grm,
                    "-r", samplesToRemove,        # Should be an empty file with NA\tNA in case nothing to remove
                    "--threads", str(threads),
                    "-n", prefix,
                    "-f", pca_dir,
                    "--selectModel", f"{pca_path}/selectModel.R",
                    "--plink1", plink,
                    "--plink2", plink2,
                    "--gcta", gcta
                    ]

In [ ]:
                   
# Define log file
logfile = f"{pca_dir}/ProjectedPCA.log"
# Run process      
with open(logfile, "w") as lf:
    subprocess.run(createProjectedPCA, cwd=f"{tools}/SAIGE_pixi/SAIGE",
                    stdout=lf, stderr=lf, check=True)
print(f"Projected PCA without reference completed successfully")

### Sparse GRM Matrix

#### Uses 64 CPUs

In [ ]:
# Create Sparse GRM
createSparseGRM =  ["pixi", "run", "Rscript", f"{saige}/createSparseGRM.R",
                    "--plinkFile", inputPfile,
                    "--outputPrefix", f"{grm_dir}/{prefix}",
                    "--relatednessCutoff", rel_cutoff,
                    "--nThreads", str(threads),
                    "--numRandomMarkerforSparseKin", str(kin_marker)]

In [ ]:
# Define log file
logfile = f"{grm_dir}/createSparseGRM.log"
# Run process      
with open(logfile, "w") as lf:
    subprocess.run(createSparseGRM, cwd=f"{tools}/SAIGE_pixi/SAIGE",
                    stdout=lf, stderr=lf, check=True)
print(f"Sparse GRM created successfully")

#### Map "DISEASE" phenotype from 1/2 to 0/1

In [ ]:
covar_step1 = pd.read_csv(f"{pca_dir}/{prefix}.tsv", sep='\t', header=0, na_values=["NA"])
covar_step1['DISEASE'] = covar_step1['DISEASE'] - 1
covar_step1 = covar_step1.rename(columns={'IID': 'ID'})
covar_step1_path = f"{pca_dir}/{prefix}.step1.tsv"
covar_step1.to_csv(covar_step1_path, sep='\t', index=False, na_rep='NA')

### SAIGE Step 1

In [ ]:
# Run Step 1
step1_fitNULLGLMM = ["pixi", "run", "Rscript", f"{saige}/step1_fitNULLGLMM.R",
                    "--plinkFile", inputPfile,
                    "--phenoFile", covar_step1_path,
                    "--phenoCol", pheno,
                    "--sampleIDColinphenoFile", "ID",
                    "--covarColList", ",".join(covariate_list),
                    "--qCovarColList", ",".join(qcovariate_list), 
                    "--isCateVarianceRatio=TRUE",
                    "--useSparseGRMtoFitNULL=TRUE",
                    "--useSparseGRMforVarRatio=TRUE",
                    "--isCateVarianceRatio=TRUE",
                    "--IsOverwriteVarianceRatioFile=TRUE",
                    "--traitType=binary",
                    "--sparseGRMFile", f"{grm_dir}/{related_file}.sparseGRM.mtx",
                    "--sparseGRMSampleIDFile", f"{grm_dir}/{related_file}.sparseGRM.mtx.sampleIDs.txt",
                    "--outputPrefix", f"{step1}/{related_file}.sparseGRM",
                    "--nThreads", str(threads)]

In [ ]:
# Define log file
logfile = f"{step1}/step1.log"
# Run process      
with open(logfile, "w") as lf:
    subprocess.run(step1_fitNULLGLMM, cwd=f"{tools}/SAIGE_pixi/SAIGE",
                    stdout=lf, stderr=lf, check=True)
print(f"Step 1 completed successfully")

### SAIGE Step 2

<div class="alert alert-block alert-info">
<b>Tip:</b> If the "CHROM" column in vcf.gz files contains "chr", make sure to add "chr" before calling for chromosomes, otherwise the error would be "No variants found" <code>"--chrom", f"chr{chrom}"</code>
</div>


In [ ]:
# Run Step 2
def parallelizeSTEP2(imputed_chr, chrom):
    step2_SPAtests = ["pixi", "run", "Rscript", f"{saige}/step2_SPAtests.R",
                    "--vcfFile", f"{imputed_chr}.vcf.gz",
                    "--vcfFileIndex", f"{imputed_chr}.vcf.gz.csi",
                    "--vcfField", "DS",
                    "--SAIGEOutputFile", f"{step2}/chr{chrom}_{prefix}",
                    "--sampleFile", f"{grm_dir}/{related_file}.sparseGRM.mtx.sampleIDs.txt",
                    "--minMAF", "0",
                    "--minMAC", "20", 
                    "--GMMATmodelFile", f"{step1}/{related_file}.sparseGRM.rda",
                    "--varianceRatioFile", f"{step1}/{related_file}.sparseGRM.varianceRatio.txt",
                    "--sparseGRMFile", f"{grm_dir}/{related_file}.sparseGRM.mtx",
                    "--sparseGRMSampleIDFile", f"{grm_dir}/{related_file}.sparseGRM.mtx.sampleIDs.txt",
                    "--SAIGEOutputFile", f"{step2}/chr{chrom}_{prefix}",
                    "--is_output_markerList_in_groupTest", "TRUE",
                    "--LOCO", "FALSE",
                    "--is_Firth_beta", "TRUE",
                    "--pCutoffforFirth", "0.1",
                    "--is_output_moreDetails", "TRUE",
                    "--AlleleOrder", "ref-first",
                    "--chrom", f"chr{chrom}",
                    "--is_fast", "TRUE", 
                    "--nThreads", "1"]

    # Define per chromosome log file
    logfile = f"{step2}/chr{chrom}_{prefix}.log"
    # Run process      
    with open(logfile, "w") as lf:
        subprocess.run(step2_SPAtests, cwd=f"{tools}/SAIGE_pixi/SAIGE",
                       stdout=lf, stderr=lf, check=True)
    print("Step 2 completed for chromosome", chrom)


In [ ]:
# Use ThreadPoolExecutor to run Step 2 in parallel
with ThreadPoolExecutor(max_workers=22) as executor:
    futures = [
        executor.submit(
            # Define arguments
            parallelizeSTEP2, f"{soft_dir}/chr{chrom}.dose", chrom
        )
        for chrom in chromosomes
    ]
    for fut in as_completed(futures):
        try:
            fut.result()
        except Exception as e:
            print(e)

## Merge sumstats

In [ ]:
dfs = []

# Append each summary stats file
for chrom in chromosomes:
    df = pl.read_csv(f"{step2}/chr{chrom}_{prefix}", separator='\t')
    dfs.append(df)

# Concatenate all dfs
sumstats = pl.concat(dfs)

# Save as a single file
sumstats.write_csv(f"{gwas_dir}/SAIGE.logistic.SEX_AGE_PC1-10.allmarkers.tsv", separator='\t')

### Plot

In [ ]:
pl_sumstats = pl.read_csv(
                f'{gwas_dir}/SAIGE.logistic.SEX_AGE_PC1-10.allmarkers.tsv', separator='\t',
                columns=['CHR', 'POS', 'MarkerID', 'Allele1', 'Allele2', 'p.value', 'BETA', 'SE']
                ).rename({
                    'Allele1': 'effect_allele',
                    'Allele2': 'other_allele',
                    'BETA': 'beta',
                    'MarkerID': 'MarkerID',
                    'p.value': 'P'
                }).with_columns([
                    pl.col('CHR').str.replace('chr', '').cast(pl.Int32),
                    pl.col('POS').cast(pl.Int32)
                ]
                )

# Conver polars to pandas for Gwaslab compatability
pd_sumstats = pl_sumstats.to_pandas()

In [ ]:
sumstats = gl.Sumstats(
                    pd_sumstats,
                    snpid="MarkerID",
                    chrom="CHR",
                    pos="POS",
                    ea="effect_allele",
                    nea="other_allele",
                    beta="beta",
                    se="SE",
                    p="P",
                    build="38"
                    )

# Plot Manhattan and QQ plot
sumstats.plot_mqq(
                mode="mqq",
                build="38",
                anno="GENENAME",
                anno_style="expand", 
                sig_level=5e-8,
                suggestive_sig_line=True,
                suggestive_sig_level=1e-5,
                # Keep font_family option to avoid errors
                font_family="sans-serif",
                save=f"{main_dir}/PLOTS/Manhattan_CATPD_SAIGE_logistic.SEX_AGE_PC1-10.png",
                # Change dpi to higher resolution for publications (>300dpi)
                save_args={"dpi":300,"facecolor":"white"}
                )